# Berth Rescheduling — Rule-Based (FCFS)

Port of Barcelona 2025 port calls. Conflicts are detected and resolved with a first-come-first-served rule per berth. No LLM is used.

## Setup

In [ ]:
%pip -q install pandas

In [ ]:
import os
import re
import time
import itertools

import pandas as pd
from IPython.display import display

## 1. Data

In [ ]:
FILE = 'escales_arribades_2025 3.csv'
COLUMNS = ['ESCALANUM', 'MOLLCODI', 'MOLLMODULS', 'ETA', 'ETD']

if not os.path.exists(FILE):
    from google.colab import files
    uploaded = files.upload()
    FILE = next(iter(uploaded))

raw = pd.read_csv(FILE, dtype='string', encoding='utf-8-sig')
raw.columns = raw.columns.str.strip().str.upper()

missing = [c for c in COLUMNS if c not in raw.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

print(f'Rows: {len(raw):,}')

In [ ]:
def modules_as_set(value):
    s = str(value).strip().replace(' ', '')
    if not re.fullmatch(r'\d+(?:-\d+)?(?:[,;]\d+(?:-\d+)?)*', s):
        raise ValueError(f'Invalid modules: {value}')
    modules = set()
    for part in re.split('[,;]', s):
        if '-' in part:
            a, b = map(int, part.split('-'))
            if a > b or b - a > 200:
                raise ValueError(f'Invalid module range: {value}')
            modules.update(range(a, b + 1))
        else:
            modules.add(int(part))
    return modules


def valid_modules(value):
    try:
        return bool(modules_as_set(value))
    except (ValueError, TypeError):
        return False


def clean_schedule(raw):
    df = raw[COLUMNS].copy()
    df.columns = ['vessel_id', 'berth', 'modules', 'eta', 'etd']
    for col in ['vessel_id', 'berth', 'modules']:
        df[col] = df[col].str.strip()
    for col in ['eta', 'etd']:
        df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')

    df = df.dropna()
    df = df[(df.vessel_id != '') & (df.berth != '') & (df.etd > df.eta)]
    df = df[df.modules.map(valid_modules)].drop_duplicates()
    df = df[~df.vessel_id.duplicated(keep=False)]
    return df.reset_index(drop=True)

In [ ]:
df = clean_schedule(raw)

print(f'Original rows: {len(raw):,}')
print(f'Clean rows:    {len(df):,}')
print(f'Berths:        {df.berth.nunique()}')
df.head()

## 2. Rescheduling

For each berth, vessels are handled in order of original ETA. A vessel keeps its berth, modules and service duration, and starts at the earliest time (not before its original ETA) when none of its modules is occupied.

In [ ]:
def reschedule(table):
    conflicts, changes = [], []

    for _, group in table.groupby('berth'):
        rows = list(group.sort_values(['eta', 'etd', 'vessel_id']).itertuples(index=False))

        active = []
        for row in rows:
            mods = modules_as_set(row.modules)
            active = [(v, m) for v, m in active if v.etd > row.eta]
            conflicts += [[v.vessel_id, row.vessel_id] for v, m in active if m & mods]
            active.append((row, mods))

        placed = []
        for row in rows:
            mods = modules_as_set(row.modules)
            duration = row.etd - row.eta
            start = row.eta
            # the vessel cannot start before every overlapping vessel on a shared module has left
            while True:
                blocking = [end for s, end, m in placed
                            if s < start + duration and start < end and m & mods]
                if not blocking:
                    break
                start = max(blocking)
            placed.append((start, start + duration, mods))

            if start != row.eta:
                changes.append({
                    'vessel_id': row.vessel_id,
                    'new_eta': start.strftime('%Y-%m-%d %H:%M:%S'),
                    'new_etd': (start + duration).strftime('%Y-%m-%d %H:%M:%S'),
                })

    return {'conflicts_identified': conflicts, 'rescheduling_changes': changes}

In [ ]:
t0 = time.perf_counter()
solution = reschedule(df)
run_time = round(time.perf_counter() - t0, 2)

print('Conflicts found:', len(solution['conflicts_identified']))
print('Vessels moved:  ', len(solution['rescheduling_changes']))
print('Run time (s):   ', run_time)
pd.DataFrame(solution['rescheduling_changes'])

## 3. Evaluation

Two port calls conflict when they share a berth, share at least one module, and their time intervals overlap (touching times do not count).

In [ ]:
def find_conflicts(table):
    pairs = set()
    for _, group in table.groupby('berth'):
        active = []
        for row in group.sort_values('eta').itertuples(index=False):
            active = [v for v in active if v.etd > row.eta]
            mods = modules_as_set(row.modules)
            for v in active:
                if mods & modules_as_set(v.modules):
                    pairs.add(tuple(sorted((v.vessel_id, row.vessel_id))))
            active.append(row)
    return pairs


def validate_changes(base, changes):
    base = base.reset_index(drop=True)
    updated = base.copy()
    errors, delays, seen = [], {}, set()
    if not isinstance(changes, list):
        return updated, ['Changes must be a list'], delays

    index = {v: i for i, v in enumerate(base.vessel_id)}
    for change in changes:
        vid = change.get('vessel_id') if isinstance(change, dict) else None
        if not isinstance(vid, str) or vid not in index:
            errors.append(f'Unknown vessel: {vid}')
            continue
        if vid in seen:
            errors.append(f'Duplicate change: {vid}')
            continue
        seen.add(vid)

        old = base.loc[index[vid]]
        if str(change.get('berth', old.berth)) != old.berth:
            errors.append(f'Berth changed: {vid}')
        if str(change.get('modules', old.modules)) != old.modules:
            errors.append(f'Modules changed: {vid}')
        try:
            eta, etd = pd.Timestamp(change['new_eta']), pd.Timestamp(change['new_etd'])
            if pd.isna(eta) or pd.isna(etd):
                raise ValueError('missing date')
            if eta < old.eta:
                errors.append(f'Earlier ETA: {vid}')
            if etd <= eta:
                errors.append(f'ETD not after ETA: {vid}')
            if etd - eta != old.etd - old.eta:
                errors.append(f'Duration changed: {vid}')
        except (KeyError, ValueError, TypeError) as e:
            errors.append(f'Invalid date for {vid}: {e}')
            continue

        delays[vid] = (eta - old.eta).total_seconds() / 60
        updated.loc[index[vid], ['eta', 'etd']] = [eta, etd]

    return updated, errors, delays


def evaluate_proposal(base, changes):
    updated, errors, delays = validate_changes(base, changes)
    valid = not errors
    return {
        'valid': valid,
        'errors': errors,
        'delays': delays,
        'schedule': updated if valid else None,
        'remaining': find_conflicts(updated) if valid else None,
    }

In [ ]:
initial = find_conflicts(df)
reported = {tuple(sorted(p)) for p in solution['conflicts_identified']}

tp = len(reported & initial)
fp = len(reported - initial)
fn = len(initial - reported)
precision = tp / len(reported) if reported else float(not initial)
recall = tp / len(initial) if initial else 1.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

evaluation = evaluate_proposal(df, solution['rescheduling_changes'])

print('Reference conflicts:', len(initial))
print(f'TP={tp}  FP={fp}  FN={fn}')
print(f'Precision={precision:.2f}  Recall={recall:.2f}  F1={f1:.2f}')
print('Valid schedule:', evaluation['valid'])
if evaluation['errors']:
    print('Constraint errors:')
    for e in evaluation['errors']:
        print(' ', e)

## 4. Success Criteria

The ten success criteria from the project evaluation document, checked on the final timetable. Delay is minimal in the sense that every delayed vessel starts exactly when the vessel blocking its module leaves (no idle gap).

In [ ]:
final = evaluation['schedule']
old = df.set_index('vessel_id')
new = final.set_index('vessel_id') if final is not None else old

brute = set()
for _, g in df.assign(mods=df.modules.map(modules_as_set)).groupby('berth'):
    for a, b in itertools.combinations(g.itertuples(index=False), 2):
        if a.eta < b.etd and b.eta < a.etd and a.mods & b.mods:
            brute.add(tuple(sorted((a.vessel_id, b.vessel_id))))

def starts_when_freed(vid):
    v = new.loc[vid]
    mods = modules_as_set(v.modules)
    same = new[(new.berth == v.berth) & (new.index != vid)]
    return any(w.etd == v.eta and w.eta < v.eta and mods & modules_as_set(w.modules)
               for w in same.itertuples())

moved = [v for v, d in evaluation['delays'].items() if d > 0]
remaining = evaluation['remaining'] or set()

eta_before_etd = (new.etd > new.eta).all()
same_berth = old.berth.equals(new.berth) and old.modules.equals(new.modules)
same_duration = ((new.etd - new.eta) == (old.etd - old.eta)).all()
no_earlier = (new.eta >= old.eta).all()
no_new = final is not None and not (remaining - initial)

criteria = [
    ('Detect berth scheduling conflicts correctly', reported == initial == brute),
    ('Resolve berth scheduling conflicts', final is not None and not remaining),
    ('Minimize the added rescheduling delay', all(starts_when_freed(v) for v in moved)),
    ('Generate a valid rescheduled timetable', evaluation['valid']),
    ('Ensure ETA is before ETD', eta_before_etd),
    ('Preserve the original berth and module assignments', same_berth),
    ('Maintain the original vessel service duration', same_duration),
    ('Avoid introducing new scheduling conflicts', no_new),
    ('Rescheduled ETA is not earlier than the original ETA', no_earlier),
    ('Validate the final timetable against all constraints',
     evaluation['valid'] and all([eta_before_etd, same_berth, same_duration, no_earlier, no_new])),
]

criteria = pd.DataFrame([(name, 'PASS' if ok else 'FAIL') for name, ok in criteria],
                        columns=['Success criterion', 'Result'])
display(criteria)
print(f"{(criteria.Result == 'PASS').sum()}/{len(criteria)} passed")

## 5. Test Cases

Normal and abnormal cases from the project evaluation document, on small synthetic tables.

In [ ]:
results = []

def test(case, description, passed):
    results.append((case, description, 'PASS' if passed else 'FAIL'))

def table(rows):
    return clean_schedule(pd.DataFrame(rows, columns=COLUMNS, dtype='string'))

def move(vid, eta, etd, **extra):
    return {'vessel_id': vid, 'new_eta': eta, 'new_etd': etd, **extra}

def score(base, changes):
    r = evaluate_proposal(base, changes)
    if not r['valid']:
        return (1, float('inf'), float('inf'), float('inf'))
    d = [x for x in r['delays'].values() if x > 0]
    return (0, len(r['remaining']), len(d), sum(d))

D = '2025-01-01 '
pair = [['A', 'B1', '1', D + '08:00', D + '12:00'],
        ['B', 'B1', '1', D + '10:00', D + '14:00']]
ok = ['OK', 'B1', '1', D + '08:00', D + '10:00']

# Normal cases
base = table(pair + [['C', 'B1', '1', D + '11:00', D + '13:00']])
r = evaluate_proposal(base, reschedule(base)['rescheduling_changes'])
test('Normal 1', 'Table with conflicts is rescheduled with fewer conflicts',
     r['valid'] and len(r['remaining']) < len(find_conflicts(base)))

base = table(pair)
s = reschedule(base)
r = evaluate_proposal(base, s['rescheduling_changes'])
test('Normal 2', 'Overlapping vessels: conflict detected and rescheduled',
     {tuple(sorted(p)) for p in s['conflicts_identified']} == {('A', 'B')} and r['remaining'] == set())

test('Normal 3', 'Same berth, different modules: no conflict',
     not find_conflicts(table([['A', 'B1', '1-2', D + '08:00', D + '12:00'],
                               ['B', 'B1', '3-4', D + '10:00', D + '14:00']])))

test('Normal 4', 'Non-overlapping times: no conflict',
     not find_conflicts(table([['A', 'B1', '1', D + '08:00', D + '10:00'],
                               ['B', 'B1', '1', D + '12:00', D + '14:00']]))
     and not find_conflicts(table([['A', 'B1', '1', D + '08:00', D + '12:00'],
                                   ['B', 'B1', '1', D + '12:00', D + '14:00']])))

options = {
    'no change': [],
    'delay B 2h': [move('B', D + '12:00', D + '16:00')],
    'delay B 5h': [move('B', D + '15:00', D + '19:00')],
    'delay A 6h': [move('A', D + '14:00', D + '18:00')],
}
best = min(options, key=lambda k: score(base, options[k]))
test('Normal 5', 'Best option chosen by conflicts, then delay',
     best == 'delay B 2h' and score(base, s['rescheduling_changes']) == score(base, options[best]))

# Abnormal cases
test('Abnormal 1', 'ETA after ETD is rejected',
     list(table([ok, ['X', 'B1', '1', D + '12:00', D + '08:00']]).vessel_id) == ['OK'])

test('Abnormal 2', 'Missing berth is rejected',
     list(table([ok, ['X', None, '1', D + '08:00', D + '10:00'],
                 ['Y', ' ', '1', D + '08:00', D + '10:00']]).vessel_id) == ['OK'])

test('Abnormal 3', 'Missing modules are rejected',
     list(table([ok, ['X', 'B1', None, D + '08:00', D + '10:00'],
                 ['Y', 'B1', 'abc', D + '08:00', D + '10:00']]).vessel_id) == ['OK'])

test('Abnormal 4', 'Duplicate records are removed',
     list(table([ok, ok, ['X', 'B1', '1', D + '12:00', D + '14:00'],
                 ['X', 'B2', '2', D + '12:00', D + '14:00']]).vessel_id) == ['OK'])

r1 = evaluate_proposal(base, [move('B', D + '12:00', D + '16:00', berth='B2')])
r2 = evaluate_proposal(base, [move('B', D + '12:00', D + '16:00', modules='5')])
test('Abnormal 5', 'Changing berth or modules is rejected',
     'Berth changed: B' in r1['errors'] and 'Modules changed: B' in r2['errors'])

r = evaluate_proposal(base, [move('A', D + '04:00', D + '08:00')])
test('Abnormal 6', 'Earlier ETA is rejected', 'Earlier ETA: A' in r['errors'])

r = evaluate_proposal(base, [move('B', D + '12:00', D + '15:00')])
test('Abnormal 7', 'Changed service duration is rejected', 'Duration changed: B' in r['errors'])

before = base.copy()
bad = [None,
       [move('ZZZ', D + '12:00', D + '16:00')],
       [move('B', 'not a date', D + '16:00')],
       [{'vessel_id': 'B', 'new_eta': D + '12:00'}]]
test('Abnormal 8', 'Invalid solution reports an error and is not applied',
     all(not r['valid'] and r['schedule'] is None
         for r in (evaluate_proposal(base, b) for b in bad)) and base.equals(before))

many = table([[f'V{i}', 'B1', '1', D + f'{8 + i // 2:02d}:{30 * (i % 2):02d}', D + '12:00'] for i in range(5)])
r = evaluate_proposal(many, reschedule(many)['rescheduling_changes'])
test('Abnormal 9', 'Many overlaps: valid solution, unresolved conflicts reported',
     r['valid'] and r['remaining'] is not None and len(find_conflicts(many)) == 10)

results = pd.DataFrame(results, columns=['Case', 'Description', 'Result'])
display(results)
print(f"{(results.Result == 'PASS').sum()}/{len(results)} passed")

## 6. Metrics

In [ ]:
if evaluation['valid']:
    remaining = evaluation['remaining']
    moved = {v: d for v, d in evaluation['delays'].items() if d != 0}
    total_delay = sum(moved.values())

    metrics = {
        'Initial conflicts': len(initial),
        'Remaining conflicts': len(remaining),
        'New conflicts': len(remaining - initial),
        'Conflict Resolution Rate (%)': round(100 * (len(initial) - len(remaining)) / len(initial), 2) if initial else 100.0,
        'Moved vessels': len(moved),
        'Average Added Delay (min)': round(total_delay / len(moved), 2) if moved else 0.0,
        'Total added delay (min)': round(total_delay, 2),
        'Schedule Deviation (min)': round(sum(abs(d) for d in moved.values()) / len(df), 2),
        'Execution time (s)': run_time,
        'Rescheduling attempts': 1,
        'LLM tokens': 0,
        'LLM cost (USD)': 0.0,
    }
    display(pd.DataFrame(metrics.items(), columns=['Metric', 'Value'], dtype=object))
else:
    print('Invalid schedule: metrics are not reported.')
    for e in evaluation['errors']:
        print(' ', e)